# Building LLM

## Goal

This notebook is a hands-on journey to build a language model from scratch.

Each version introduces one new concept, allowing the model to evolve step by step while practicing language-model development.

---

## Version 5

In this version, we extend the neural character language model with embeddings and a fixed context window.

Instead of using only one character to predict the next character, the model now uses multiple previous characters as context.

Character IDs are mapped to small trainable embedding vectors.

This version introduces learned embeddings and longer context while keeping the model small and easy to inspect.

## 1. Imports and Configuration

The Python standard library configures the execution environment before TensorFlow is imported.

GPU execution is disabled because this small model runs efficiently on the CPU and does not require CUDA. Low-level TensorFlow logs are suppressed to keep the notebook output clean.

TensorFlow provides tensor operations, trainable variables and automatic differentiation.

NumPy remains useful for reproducible data shuffling and sampling, while TensorFlow performs the model calculations and training.

The configuration collects the values that control the experiment.

`CONTEXT_LENGTH` defines how many previous characters are used to predict the next character.

`EMBEDDING_DIM` defines the size of the trainable vector used to represent each character.

An explicit seed makes weight initialization, data splitting, mini-batch shuffling and text generation reproducible.

In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import tensorflow as tf

tf.config.set_visible_devices([], "GPU")

In [2]:
SEED = 42
TRAIN_FRACTION = 0.8
BATCH_SIZE = 32
LEARNING_RATE = 1.0
EPOCHS = 100

CONTEXT_LENGTH = 4
EMBEDDING_DIM = 8

tf.keras.utils.set_random_seed(SEED)

## 2. Training Data

### Training text

The English corpus is included directly in the notebook. It provides the text from which the model learns character patterns.

The corpus is kept unchanged from Version 4 so that the effect of the new model representation can be observed without changing the training data.

In [3]:
corpus = 'language models learn patterns from text.\na small model predicts what character may come next.\nwe begin with counting because counting is easy to inspect.\nthe model sees letters, spaces, and punctuation.\neach prediction comes from examples found in the training text.\nsimple systems help us understand more advanced systems.\nlater versions will learn parameters with neural networks.\nclear experiments make machine learning easier to study.'

print(corpus)
print("Characters:", len(corpus))

language models learn patterns from text.
a small model predicts what character may come next.
we begin with counting because counting is easy to inspect.
the model sees letters, spaces, and punctuation.
each prediction comes from examples found in the training text.
simple systems help us understand more advanced systems.
later versions will learn parameters with neural networks.
clear experiments make machine learning easier to study.
Characters: 440


### Vocabulary

The vocabulary is the set of symbols the model can represent. Because this is a character model, every letter, space, punctuation mark and newline is a token.
The neural model also assigns an integer identifier to every character so that characters can be represented numerically.

In this version, these identifiers will also be used to look up trainable embedding vectors.

In [4]:
vocabulary = sorted(set(corpus))
vocabulary_size = len(vocabulary)

character_to_id = {character: index for index, character in enumerate(vocabulary)}
id_to_character = {index: character for character, index in character_to_id.items()}

print("Vocabulary size:", vocabulary_size)
print("Vocabulary:", repr("".join(vocabulary)))
print("First mappings:", list(character_to_id.items())[:10])

Vocabulary size: 27
Vocabulary: '\n ,.abcdefghiklmnoprstuvwxy'
First mappings: [('\n', 0), (' ', 1), (',', 2), ('.', 3), ('a', 4), ('b', 5), ('c', 6), ('d', 7), ('e', 8), ('f', 9)]


### Context windows and numerical encoding

Each character is converted into its integer identifier.

For the text `modeling`:

`modeling`  
↓  
`[15, 17, 7, 8, 14, 12, 16, 10]`

The model now uses multiple previous characters to predict the next character.

With a context length of 4, four adjacent identifiers form each input context:

- `mode -> l` becomes `[15, 17, 7, 8] -> 14`
- `odel -> i` becomes `[17, 7, 8, 14] -> 12`
- `deli -> n` becomes `[7, 8, 14, 12] -> 16`
- `elin -> g` becomes `[8, 14, 12, 16] -> 10`

The four identifiers in each context are the input. The following identifier is the target to predict.

As the context window moves through the text, the model learns to predict the next character using the four previous characters instead of only one.

The input identifiers will later be mapped to trainable embedding vectors.

During generation, predicted identifiers are converted back into characters:

`[15, 17, 7, 8, 14, 12, 16, 10]`  
↓  
`modeling`

In [5]:
examples = [
    (corpus[index:index + CONTEXT_LENGTH], corpus[index + CONTEXT_LENGTH])
    for index in range(len(corpus) - CONTEXT_LENGTH)
]

input_ids = np.array([
    [character_to_id[character] for character in context]
    for context, _ in examples
], dtype=np.int32)

target_ids = np.array([
    character_to_id[target]
    for _, target in examples
], dtype=np.int32)

print("Number of examples:", len(examples))
print("Input shape:", input_ids.shape)
print("Target shape:", target_ids.shape)
print("First 8 examples:", examples[:8])
print("First 8 input IDs:")
print(input_ids[:8])
print("First 8 target IDs:", target_ids[:8])

Number of examples: 436
Input shape: (436, 4)
Target shape: (436,)
First 8 examples: [('lang', 'u'), ('angu', 'a'), ('ngua', 'g'), ('guag', 'e'), ('uage', ' '), ('age ', 'm'), ('ge m', 'o'), ('e mo', 'd')]
First 8 input IDs:
[[14  4 16 10]
 [ 4 16 10 22]
 [16 10 22  4]
 [10 22  4 10]
 [22  4 10  8]
 [ 4 10  8  1]
 [10  8  1 15]
 [ 8  1 15 17]]
First 8 target IDs: [22  4 10  8  1 15 17  7]


## 3. Neural Model

### Trainable embeddings and output weights

In this version, each character identifier is used to select a small trainable vector from an embedding matrix.

For a context of four characters, the model retrieves four embedding vectors:

`[15, 17, 7, 8]`  
↓  
`4 embedding vectors`

Each embedding contains `EMBEDDING_DIM` learned values.

The four vectors are then concatenated into a single representation of the context.

This representation is multiplied by a trainable output weight matrix to produce one logit for every possible next character.

Both the embedding matrix and the output weight matrix are learned during training.

In [6]:
inputs = tf.convert_to_tensor(input_ids, dtype=tf.int32)

model_random = tf.random.Generator.from_seed(SEED)

embedding_matrix = tf.Variable(
    model_random.normal(
        shape=(vocabulary_size, EMBEDDING_DIM),
        mean=0.0,
        stddev=0.01,
        dtype=tf.float32
    )
)

output_weights = tf.Variable(
    model_random.normal(
        shape=(CONTEXT_LENGTH * EMBEDDING_DIM, vocabulary_size),
        mean=0.0,
        stddev=0.01,
        dtype=tf.float32
    )
)

example_embeddings = tf.gather(embedding_matrix, inputs[:1])
example_context = tf.reshape(
    example_embeddings,
    (1, CONTEXT_LENGTH * EMBEDDING_DIM)
)

trainable_parameters = (tf.size(embedding_matrix) + tf.size(output_weights))

print("Input tensor shape:", inputs.shape)
print("Embedding matrix shape:", embedding_matrix.shape)
print("Embedded context shape:", example_embeddings.shape)
print("Flattened context shape:", example_context.shape)
print("Output weight matrix shape:", output_weights.shape)
print("Trainable parameters:", trainable_parameters.numpy())

Input tensor shape: (436, 4)
Embedding matrix shape: (27, 8)
Embedded context shape: (1, 4, 8)
Flattened context shape: (1, 32)
Output weight matrix shape: (32, 27)
Trainable parameters: 1080


### Training and validation split

The examples are divided into two separate groups:

- the training set is used to update the model parameters;
- the validation set is used to measure the loss on examples that do not update the parameters.

Each input example now contains a context window of multiple character identifiers instead of a single character.

The indices are shuffled with a local random generator, making the split reproducible.

In [7]:
split_random = np.random.default_rng(SEED)

indices = split_random.permutation(len(inputs))
split_position = int(len(indices) * TRAIN_FRACTION)

train_indices = indices[:split_position]
validation_indices = indices[split_position:]

train_inputs = tf.gather(inputs, train_indices)
train_targets = tf.gather(target_ids, train_indices)

validation_inputs = tf.gather(inputs, validation_indices)
validation_targets = tf.gather(target_ids, validation_indices)

print("Training examples:", len(train_inputs))
print("Validation examples:", len(validation_inputs))
print("Training input shape:", train_inputs.shape)
print("Validation input shape:", validation_inputs.shape)

Training examples: 348
Validation examples: 88
Training input shape: (348, 4)
Validation input shape: (88, 4)


### Softmax probabilities

TensorFlow provides `tf.nn.softmax` to convert logits into probabilities.

- every probability is between 0 and 1;
- the probabilities for one input sum to 1;
- higher logits produce higher probabilities.

TensorFlow handles the numerical stability of this operation internally.

In [8]:
def softmax(logits):
    return tf.nn.softmax(logits, axis=1)

### Cross-entropy loss

Cross-entropy measures how much probability the model assigns to the correct next character.

Before calculating the loss, the context identifiers are mapped to their embedding vectors. The embeddings are concatenated and multiplied by the output weight matrix to produce the logits.

TensorFlow calculates the cross-entropy directly from the logits and integer target IDs using a numerically stable operation.

A high probability for the correct character produces a low loss. Training updates both the embeddings and the output weights to reduce this value.

In [9]:
def calculate_loss(embedding_matrix, output_weights, inputs, targets):
    embeddings = tf.gather(embedding_matrix, inputs)

    context_vectors = tf.reshape(embeddings, (tf.shape(inputs)[0], CONTEXT_LENGTH * EMBEDDING_DIM))

    logits = tf.matmul(context_vectors, output_weights)

    example_losses = tf.nn.sparse_softmax_cross_entropy_with_logits(labels=targets, logits=logits)

    return tf.reduce_mean(example_losses)

In [10]:
initial_train_loss = calculate_loss(
    embedding_matrix,
    output_weights,
    train_inputs,
    train_targets
)

initial_validation_loss = calculate_loss(
    embedding_matrix,
    output_weights,
    validation_inputs,
    validation_targets
)

print("Initial train loss:", initial_train_loss.numpy())
print("Initial validation loss:", initial_validation_loss.numpy())

Initial train loss: 3.2958233
Initial validation loss: 3.295813


### Training with mini-batch gradient descent

Training remains organized into epochs.

During every epoch:

1. the training examples are shuffled;
2. the examples are divided into mini-batches;
3. `tf.GradientTape` records the model calculations;
4. TensorFlow calculates the gradients automatically;
5. the embedding matrix and output weights are updated with gradient descent;
6. training and validation loss are measured.

The validation examples are never used to update the model parameters.

The training process remains unchanged in principle. The main difference from Version 4 is that the model now learns both the character embeddings and the output weights.

### Mini-batches

A mini-batch is a small group of training examples.

The model updates its parameters after every mini-batch instead of processing all training examples together.

The final mini-batch may contain fewer examples than the configured batch size.

In [11]:
def create_batches(inputs, targets, batch_size):
    for start in range(0, len(inputs), batch_size):
        end = start + batch_size

        batch_inputs = inputs[start:end]
        batch_targets = targets[start:end]

        yield batch_inputs, batch_targets

In [12]:
def train_model(
    train_inputs,
    train_targets,
    validation_inputs,
    validation_targets,
    initial_embedding_matrix,
    initial_output_weights,
    learning_rate=1.0,
    batch_size=32,
    epochs=100,
    seed=42,
    print_every=10
):
    trained_embedding_matrix = tf.Variable(initial_embedding_matrix)
    trained_output_weights = tf.Variable(initial_output_weights)

    training_random = np.random.default_rng(seed)

    train_loss_history = []
    validation_loss_history = []

    for epoch in range(epochs + 1):
        train_loss = float(
            calculate_loss(
                trained_embedding_matrix,
                trained_output_weights,
                train_inputs,
                train_targets
            ).numpy()
        )

        validation_loss = float(
            calculate_loss(
                trained_embedding_matrix,
                trained_output_weights,
                validation_inputs,
                validation_targets
            ).numpy()
        )

        train_loss_history.append(train_loss)
        validation_loss_history.append(validation_loss)

        if print_every is not None and epoch % print_every == 0:
            print(
                f"Epoch {epoch:3d} | "
                f"Train loss: {train_loss:.4f} | "
                f"Validation loss: {validation_loss:.4f}"
            )

        if epoch == epochs:
            break

        shuffled_indices = training_random.permutation(len(train_inputs))

        shuffled_inputs = tf.gather(train_inputs, shuffled_indices)
        shuffled_targets = tf.gather(train_targets, shuffled_indices)

        for batch_inputs, batch_targets in create_batches(
            shuffled_inputs,
            shuffled_targets,
            batch_size
        ):
            with tf.GradientTape() as tape:
                batch_loss = calculate_loss(
                    trained_embedding_matrix,
                    trained_output_weights,
                    batch_inputs,
                    batch_targets
                )

            embedding_gradient, output_gradient = tape.gradient(
                batch_loss,
                [trained_embedding_matrix, trained_output_weights]
            )

            trained_embedding_matrix.assign_sub(learning_rate * tf.convert_to_tensor(embedding_gradient))

            trained_output_weights.assign_sub(learning_rate * output_gradient)

    return (
        trained_embedding_matrix,
        trained_output_weights,
        train_loss_history,
        validation_loss_history
    )

In [13]:
(
    trained_embedding_matrix,
    trained_output_weights,
    train_loss_history,
    validation_loss_history
) = train_model(
    train_inputs,
    train_targets,
    validation_inputs,
    validation_targets,
    embedding_matrix,
    output_weights,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    seed=SEED
)

final_train_loss = train_loss_history[-1]
final_validation_loss = validation_loss_history[-1]

best_validation_epoch = int(np.argmin(validation_loss_history))
best_validation_loss = validation_loss_history[best_validation_epoch]

print()
print("Initial train loss:", initial_train_loss.numpy())
print("Final train loss:", final_train_loss)
print("Initial validation loss:", initial_validation_loss.numpy())
print("Final validation loss:", final_validation_loss)
print("Best validation loss:", best_validation_loss)
print("Best validation epoch:", best_validation_epoch)

Epoch   0 | Train loss: 3.2958 | Validation loss: 3.2958
Epoch  10 | Train loss: 2.4069 | Validation loss: 3.0056
Epoch  20 | Train loss: 1.4427 | Validation loss: 2.9843
Epoch  30 | Train loss: 1.0242 | Validation loss: 3.3867
Epoch  40 | Train loss: 0.8253 | Validation loss: 3.9169
Epoch  50 | Train loss: 0.7771 | Validation loss: 4.4448
Epoch  60 | Train loss: 0.6534 | Validation loss: 4.9234
Epoch  70 | Train loss: 0.6217 | Validation loss: 5.5643
Epoch  80 | Train loss: 0.5877 | Validation loss: 5.8272
Epoch  90 | Train loss: 0.5607 | Validation loss: 6.0404
Epoch 100 | Train loss: 0.5043 | Validation loss: 6.6350

Initial train loss: 3.2958233
Final train loss: 0.5043221116065979
Initial validation loss: 3.295813
Final validation loss: 6.635019302368164
Best validation loss: 2.8413312435150146
Best validation epoch: 14


### Reading the losses

Training loss measures the examples used to update the model.

Validation loss measures separate examples that do not update the model parameters.

In this run, the validation loss reaches its lowest value at epoch 14 and then increases substantially, while the training loss continues to decrease.

This shows clear overfitting: the larger context-based model learns the small training set very well but becomes less effective on the validation examples.

The best validation epoch is recorded for analysis, but the training procedure does not implement early stopping or restore the best parameters. Text generation uses the final parameters from epoch 100.

### Learned probabilities

After training, the model can produce a next-character probability distribution for any valid context.

Unlike Version 4, the prediction now depends on multiple previous characters instead of a single character.

The context identifiers are mapped to their learned embeddings, concatenated and transformed into logits.

Softmax converts the logits into probabilities for every possible next character.

The probabilities are learned from the training set. The validation set measures the model without updating its parameters.

In [14]:
def next_character_probabilities(
    embedding_matrix,
    output_weights,
    context
):
    if len(context) != CONTEXT_LENGTH:
        raise ValueError(
            f"The context must contain exactly {CONTEXT_LENGTH} characters."
        )

    if any(character not in character_to_id for character in context):
        raise ValueError("The context contains a character not in the vocabulary.")

    context_ids = tf.constant([[character_to_id[character] for character in context]], dtype=tf.int32)

    embeddings = tf.gather(embedding_matrix, context_ids)

    context_vector = tf.reshape(embeddings, (1, CONTEXT_LENGTH * EMBEDDING_DIM))

    logits = tf.matmul(context_vector, output_weights)

    probabilities = softmax(logits)[0].numpy()

    return {
        id_to_character[index]: float(probability)
        for index, probability in enumerate(probabilities)
    }

In [15]:
probabilities_after_mode = next_character_probabilities(
    trained_embedding_matrix,
    trained_output_weights,
    "mode"
)

most_likely_after_mode = sorted(
    probabilities_after_mode.items(),
    key=lambda item: item[1],
    reverse=True
)[:10]

print("Most likely characters after 'mode':")

for character, probability in most_likely_after_mode:
    print(f"{repr(character):>4}: {probability:.4f}")

print("Total:", sum(probabilities_after_mode.values()))

Most likely characters after 'mode':
 'l': 0.8659
 ' ': 0.0584
 'r': 0.0461
 'c': 0.0266
 's': 0.0024
 'i': 0.0005
 '.': 0.0000
 'e': 0.0000
 'f': 0.0000
 'o': 0.0000
Total: 1.0000001844501494


## 4. Generator

### Sample the next character

The current context contains the previous `CONTEXT_LENGTH` characters.

Their identifiers are mapped to learned embedding vectors, concatenated and transformed into logits.

Softmax converts the logits into probabilities, and the next character is sampled from that distribution.

After sampling, the new character is added to the context and the oldest character is removed.

This creates a sliding context window that moves forward one character at a time.

In [16]:
def sample_next_character(embedding_matrix, output_weights, context_ids, random_generator):
    context_tensor = tf.constant([context_ids], dtype=tf.int32)

    embeddings = tf.gather(embedding_matrix, context_tensor)

    context_vector = tf.reshape(embeddings, (1, CONTEXT_LENGTH * EMBEDDING_DIM))

    logits = tf.matmul(
        context_vector,
        output_weights
    )

    probabilities = softmax(logits)[0].numpy()

    return random_generator.choice(vocabulary_size, p=probabilities)

In [17]:
demo_random = np.random.default_rng(SEED)

mode_ids = [character_to_id[character] for character in "mode"]

sampled_ids = [
    sample_next_character(
        trained_embedding_matrix,
        trained_output_weights,
        mode_ids,
        demo_random
    )
    for _ in range(5)
]

print(
    "Five samples after 'mode':",
    [id_to_character[index] for index in sampled_ids]
)

Five samples after 'mode': ['l', 'l', 'l', 'l', 'l']


### Generate text

Generation repeats the same autoregressive loop:

1. read the current context;
2. use its learned embeddings to calculate next-character probabilities;
3. sample the next character;
4. append it to the generated text;
5. remove the oldest character from the context and add the new character.

For example:

`mode -> l`

The context then moves forward:

`odel -> ?`

and continues one character at a time.

The seed makes the example reproducible.

Unlike Version 4, each prediction now uses multiple previous characters instead of only the most recent character.

In [18]:
def generate_text(
    embedding_matrix,
    output_weights,
    start_context="mode",
    length=200,
    seed=42
):
    if len(start_context) != CONTEXT_LENGTH:
        raise ValueError(f"The start context must contain exactly {CONTEXT_LENGTH} characters.")

    if length < CONTEXT_LENGTH:
        raise ValueError(f"Length must be at least {CONTEXT_LENGTH}.")

    if any(character not in character_to_id for character in start_context):
        raise ValueError("The start context contains a character not in the vocabulary.")

    random_generator = np.random.default_rng(seed)

    generated_ids = [character_to_id[character] for character in start_context]

    while len(generated_ids) < length:
        context_ids = generated_ids[-CONTEXT_LENGTH:]

        next_id = sample_next_character(
            embedding_matrix,
            output_weights,
            context_ids,
            random_generator
        )

        generated_ids.append(next_id)

    return "".join(id_to_character[index] for index in generated_ids)

In [19]:
generated_text = generate_text(
    trained_embedding_matrix,
    trained_output_weights,
    start_context="mode",
    length=300,
    seed=SEED
)

print(generated_text)

model tratne seactudy.ntudy.ndmampinicima ledir to ine begtne seactueg cama niticing is eara natime le text.
wimes fromane worh neuse nsper verh wouncong wimespersmaynimmel seactimase nspersiansimes from the coune nspersians makecpund is will leare pr model pred framate n.
ame pr moy come learn pare


## 5. Tests

These assertions verify the context windows, TensorFlow tensors, trainable embeddings, output weights, data split, model shapes, probability distributions, training behavior and reproducibility.

The training procedure is repeated from the same initial parameters and with the same seed to verify that TensorFlow produces the same embeddings, output weights and loss histories.

The tests also verify that generation is reproducible when the same starting context and seed are used.

In [20]:
assert len(examples) == len(corpus) - CONTEXT_LENGTH
assert len(input_ids) == len(examples)
assert len(target_ids) == len(examples)

assert input_ids.shape == (len(examples), CONTEXT_LENGTH)
assert target_ids.shape == (len(examples),)

assert examples[0][0] == corpus[:CONTEXT_LENGTH]
assert examples[0][1] == corpus[CONTEXT_LENGTH]

assert examples[1][0] == corpus[1:1 + CONTEXT_LENGTH]
assert examples[1][1] == corpus[1 + CONTEXT_LENGTH]

assert tf.is_tensor(inputs)

assert isinstance(embedding_matrix, tf.Variable)
assert isinstance(output_weights, tf.Variable)

assert isinstance(trained_embedding_matrix, tf.Variable)
assert isinstance(trained_output_weights, tf.Variable)

assert inputs.shape == (len(examples), CONTEXT_LENGTH)

assert embedding_matrix.shape == (vocabulary_size, EMBEDDING_DIM)

assert output_weights.shape == (CONTEXT_LENGTH * EMBEDDING_DIM, vocabulary_size)

assert trained_embedding_matrix.shape == embedding_matrix.shape
assert trained_output_weights.shape == output_weights.shape

assert len(train_inputs) + len(validation_inputs) == len(inputs)
assert len(train_inputs) == len(train_targets)
assert len(validation_inputs) == len(validation_targets)

assert len(np.intersect1d(train_indices, validation_indices)) == 0

assert len(train_loss_history) == EPOCHS + 1
assert len(validation_loss_history) == EPOCHS + 1

assert final_train_loss < float(initial_train_loss.numpy())

assert np.all(np.isfinite(train_loss_history))
assert np.all(np.isfinite(validation_loss_history))

probability_total = sum(
    next_character_probabilities(
        trained_embedding_matrix,
        trained_output_weights,
        "mode"
    ).values()
)

assert abs(probability_total - 1.0) < 1e-6

gradient_inputs = train_inputs[:BATCH_SIZE]
gradient_targets = train_targets[:BATCH_SIZE]

with tf.GradientTape() as tape:
    gradient_loss = calculate_loss(
        embedding_matrix,
        output_weights,
        gradient_inputs,
        gradient_targets
    )

embedding_gradient, output_gradient = tape.gradient(
    gradient_loss,
    [embedding_matrix, output_weights]
)

assert embedding_gradient is not None
assert output_gradient is not None

dense_embedding_gradient = tf.convert_to_tensor(embedding_gradient)

assert dense_embedding_gradient.shape == embedding_matrix.shape
assert output_gradient.shape == output_weights.shape

assert np.all(np.isfinite(dense_embedding_gradient.numpy()))

assert np.all(np.isfinite(output_gradient.numpy()))

(
    repeated_embedding_matrix,
    repeated_output_weights,
    repeated_train_history,
    repeated_validation_history
) = train_model(
    train_inputs,
    train_targets,
    validation_inputs,
    validation_targets,
    embedding_matrix,
    output_weights,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    seed=SEED,
    print_every=None
)

assert np.allclose(trained_embedding_matrix.numpy(), repeated_embedding_matrix.numpy())

assert np.allclose(trained_output_weights.numpy(), repeated_output_weights.numpy())

assert np.allclose(train_loss_history, repeated_train_history)

assert np.allclose(validation_loss_history, repeated_validation_history)

assert not np.allclose(embedding_matrix.numpy(), trained_embedding_matrix.numpy())

assert not np.allclose(output_weights.numpy(), trained_output_weights.numpy())

first_generation = generate_text(
    trained_embedding_matrix,
    trained_output_weights,
    start_context="mode",
    length=30,
    seed=10
)

second_generation = generate_text(
    trained_embedding_matrix,
    trained_output_weights,
    start_context="mode",
    length=30,
    seed=10
)

assert first_generation == second_generation
assert len(first_generation) == 30
assert first_generation.startswith("mode")

print("All checks passed.")

All checks passed.


## Notes

- The model remains a character-level neural language model.
- Each prediction now uses a fixed context of multiple previous characters instead of a single character.
- Character identifiers are mapped to small trainable embedding vectors.
- The embeddings in the context are concatenated into a single context representation.
- A trainable output weight matrix converts the context representation into next-character logits.
- Both the embedding matrix and the output weights are learned with `tf.GradientTape`.
- The examples are divided into reproducible training and validation sets.
- Training examples are shuffled at the beginning of every epoch.
- Mini-batches are used to update the model parameters.
- Training loss measures performance on the examples used for learning.
- Validation loss measures performance on separate examples that never update the model parameters.
- The same seed reproduces initialization, data splitting, training and generation.
- The generator remains autoregressive and now uses a sliding context window.
- The tests verify context windows, embeddings, model shapes, gradients, training behavior and reproducibility.

The model can now use information from several previous characters, but the context still has a fixed length and is processed as a single flattened representation.

Future versions will introduce new components and gradually evolve the architecture.